In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, f1_score, roc_curve


In [2]:
fold_1_train = pd.read_csv('/Users/sreenithi/Downloads/data/preprocessed/fold_1_train.csv')
fold_2_train = pd.read_csv('/Users/sreenithi/Downloads/data/preprocessed/fold_2_train.csv')
fold_3_train = pd.read_csv('/Users/sreenithi/Downloads/data/preprocessed/fold_3_train.csv')
fold_4_train = pd.read_csv('/Users/sreenithi/Downloads/data/preprocessed/fold_4_train.csv')
fold_5_train = pd.read_csv('/Users/sreenithi/Downloads/data/preprocessed/fold_5_train.csv')

fold_1_val = pd.read_csv('/Users/sreenithi/Downloads/data/preprocessed/fold_1_val.csv')
fold_2_val = pd.read_csv('/Users/sreenithi/Downloads/data/preprocessed/fold_2_val.csv')
fold_3_val = pd.read_csv('/Users/sreenithi/Downloads/data/preprocessed/fold_3_val.csv')
fold_4_val = pd.read_csv('/Users/sreenithi/Downloads/data/preprocessed/fold_4_val.csv')
fold_5_val = pd.read_csv('/Users/sreenithi/Downloads/data/preprocessed/fold_5_val.csv')

In [3]:
train_folds = [fold_1_train, fold_2_train, fold_3_train, fold_4_train, fold_5_train]
val_folds = [fold_1_val, fold_2_val, fold_3_val, fold_4_val, fold_5_val]

In [4]:
# Concatenate SMOTEd folds 1-4 for training
training_data = pd.concat([fold_1_train, fold_2_train, fold_3_train, fold_4_train])
X_train = training_data.drop(columns='fraud_bool')
y_train = training_data['fraud_bool']

# Use fold 5 as validation
X_val = fold_5_val.drop(columns='fraud_bool') 
y_val = fold_5_val['fraud_bool']

In [5]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Feature selection (selecting 20 as default — this will be tuned later)
k_best = 30
selector = SelectKBest(score_func=f_classif, k=k_best)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_val_selected = selector.transform(X_val_scaled)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/feature_selection/_univariate_selection.py:112: UserWarning: Features [25] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/feature_selection/_univariate_selection.py:113: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [6]:
import optuna
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import tensorflow as tf
import numpy as np

# Compute class weights
class_weights_array = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: class_weights_array[i] for i in range(len(class_weights_array))}

# Use 10% of training data for tuning
X_tune, _, y_tune, _ = train_test_split(X_train_selected, y_train, test_size=0.9, stratify=y_train, random_state=42)

# Sequence creation
def create_sequences(X, y, n_steps=7):
    Xs, ys = [], []
    for i in range(len(X) - n_steps):
        Xs.append(X[i:i+n_steps])
        ys.append(y[i+n_steps])
    return np.array(Xs), np.array(ys)

def objective(trial):
    tf.keras.backend.clear_session()

    # Hyperparameters
    gru_units = trial.suggest_int('gru_units', 64, 256)
    dropout_rate = trial.suggest_float('dropout_rate', 0.3, 0.5)
    n_steps = trial.suggest_int('n_steps', 7, 15)
    batch_size = trial.suggest_int('batch_size', 32, 64)

    # Sequences
    X_seq, y_seq = create_sequences(X_tune, y_tune.values, n_steps)
    
    model = Sequential([
        GRU(gru_units, activation='relu', input_shape=(n_steps, X_seq.shape[2])),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy')

    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.fit(X_seq, y_seq, epochs=30, batch_size=batch_size, 
              validation_split=0.2, class_weight=class_weight_dict,
              callbacks=[early_stop], verbose=0)

    preds = model.predict(X_seq).flatten()
    preds_binary = (preds > 0.5).astype(int)
    return f1_score(y_seq, preds_binary)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

best_params = study.best_params
print("Best hyperparameters:", best_params)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-04-10 20:35:00,107] A new study created in memory with name: no-name-66ebc5d5-180a-4916-b3ab-a220e935c037
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


17871/17871 ━━━━━━━━━━━━━━━━━━━━ 64s 4ms/step


[I 2025-04-10 20:53:25,981] Trial 0 finished with value: 0.4124128213572563 and parameters: {'gru_units': 189, 'dropout_rate': 0.34709894330305663, 'n_steps': 14, 'batch_size': 58}. Best is trial 0 with value: 0.4124128213572563.
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# Recreate selected features with best k
selector = SelectKBest(score_func=f_classif, k=best_params['k_best'] if 'k_best' in best_params else k_best)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_val_selected = selector.transform(X_val_scaled)

# Create final sequences
X_train_seq, y_train_seq = create_sequences(X_train_selected, y_train.values, best_params['n_steps'])
X_val_seq, y_val_seq = create_sequences(X_val_selected, y_val.values, best_params['n_steps'])

# Model definition
final_model = Sequential([
    GRU(best_params['gru_units'], activation='relu', input_shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
    Dropout(best_params['dropout_rate']),
    Dense(1, activation='sigmoid')
])

final_model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy')

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train model
final_model.fit(X_train_seq, y_train_seq, epochs=50, batch_size=best_params['batch_size'],
                validation_data=(X_val_seq, y_val_seq), class_weight=class_weight_dict,
                callbacks=[early_stop], verbose=1)

# Predictions
y_val_probs = final_model.predict(X_val_seq).flatten()
y_val_preds = (y_val_probs > 0.5).astype(int)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

adjusted_y_val = y_val.values[best_params['n_steps']:]

print("Classification Report on Validation Set:")
print(classification_report(adjusted_y_val, y_val_preds))

print("Confusion Matrix:")
print(confusion_matrix(adjusted_y_val, y_val_preds))

roc_auc = roc_auc_score(adjusted_y_val, y_val_probs)
print(f'ROC AUC on Validation Set: {roc_auc:.2f}')

f1 = f1_score(adjusted_y_val, y_val_preds)
print(f'F1 Score on Validation Set: {f1:.4f}')

# ROC Curve
y_train_probs = final_model.predict(X_train_seq).flatten()
adjusted_y_train = y_train.values[best_params['n_steps']:]
fpr_train, tpr_train, _ = roc_curve(adjusted_y_train, y_train_probs)
fpr_val, tpr_val, _ = roc_curve(adjusted_y_val, y_val_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr_train, tpr_train, label=f'Train ROC (AUC = {roc_auc_score(adjusted_y_train, y_train_probs):.2f})')
plt.plot(fpr_val, tpr_val, label=f'Validation ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for GRU Model')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()
